In [4]:
!pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import pandas as pd

raw_fields_pd = pd.read_excel("trv_api_fields_raw.xlsx", sheet_name='raw')
raw_fields_pd.head(30)

,Name,Display name,Type
0,type,Symbol Type,text
1,is_primary,Primary Listing,bool
2,active_symbol,Current trading day,bool
3,index,Index,text
4,earnings_per_share_basic_ttm,Basic EPS (TTM),fundamental_price
5,change,NaN,NaN
6,change,NaN,NaN
7,change|1,NaN,NaN
8,change|5,NaN,NaN
9,change|15,NaN,NaN


In [61]:
raw_fields_pd['Group Name'] = raw_fields_pd['Name'].apply(lambda s: '|'.join(str(s).split('|')[:-1]) if '|' in s else s)
raw_fields_pd.head(30)

,Name,Display name,Type,Group Name
0,type,Symbol Type,text,type
1,is_primary,Primary Listing,bool,is_primary
2,active_symbol,Current trading day,bool,active_symbol
3,index,Index,text,index
4,earnings_per_share_basic_ttm,Basic EPS (TTM),fundamental_price,earnings_per_share_basic_ttm
5,change,NaN,NaN,change
6,change,NaN,NaN,change
7,change|1,NaN,NaN,change
8,change|5,NaN,NaN,change
9,change|15,NaN,NaN,change


In [78]:
detailed_fields_pd = raw_fields_pd.copy()
detailed_fields_pd['is_same_group'] = detailed_fields_pd['Group Name'] == detailed_fields_pd['Group Name'].shift(1)
detailed_fields_pd['is_group_description'] = (~detailed_fields_pd['is_same_group']) & detailed_fields_pd['is_same_group'].shift(1)
detailed_fields_pd['is_group_header'] = (
        (
                (~detailed_fields_pd['is_same_group']) & (detailed_fields_pd['is_same_group'].shift(-1).fillna(False))
        ) |
        (
                (detailed_fields_pd['is_same_group']) & (~detailed_fields_pd['is_same_group'].shift(1).fillna(True))
        )
)

detailed_fields_pd.loc[detailed_fields_pd['is_group_description'], 'Type'] = detailed_fields_pd.loc[detailed_fields_pd['is_group_description'], 'Display name']
detailed_fields_pd.loc[detailed_fields_pd['is_group_description'], 'Display name'] = detailed_fields_pd.loc[detailed_fields_pd['is_group_description'], 'Name']

detailed_fields_pd = detailed_fields_pd.bfill()
detailed_fields_pd.head(30)

C:\Users\Shachar\AppData\Local\Temp\ipykernel_23776\2755360455.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (~detailed_fields_pd['is_same_group']) & (detailed_fields_pd['is_same_group'].shift(-1).fillna(False))
C:\Users\Shachar\AppData\Local\Temp\ipykernel_23776\2755360455.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (detailed_fields_pd['is_same_group']) & (~detailed_fields_pd['is_same_group'].shift(1).fillna(True))


,Name,Display name,Type,Group Name,is_same_group,is_group_description,is_group_header
0,type,Symbol Type,text,type,False,False,False
1,is_primary,Primary Listing,bool,is_primary,False,False,False
2,active_symbol,Current trading day,bool,active_symbol,False,False,False
3,index,Index,text,index,False,False,False
4,earnings_per_share_basic_ttm,Basic EPS (TTM),fundamental_price,earnings_per_share_basic_ttm,False,False,False
5,change,Change %,percent,change,False,False,True
6,change,Change %,percent,change,True,False,True
7,change|1,Change %,percent,change,True,False,False
8,change|5,Change %,percent,change,True,False,False
9,change|15,Change %,percent,change,True,False,False


In [81]:
processed_fields_pd = detailed_fields_pd.loc[
        ~detailed_fields_pd['is_group_description'] & ~detailed_fields_pd['is_group_header']
][
    [
        'Name',
        'Display name',
        'Type',
        'Group Name'
    ]
]

processed_fields_pd.head(30)

,Name,Display name,Type,Group Name
0,type,Symbol Type,text,type
1,is_primary,Primary Listing,bool,is_primary
2,active_symbol,Current trading day,bool,active_symbol
3,index,Index,text,index
4,earnings_per_share_basic_ttm,Basic EPS (TTM),fundamental_price,earnings_per_share_basic_ttm
7,change|1,Change %,percent,change
8,change|5,Change %,percent,change
9,change|15,Change %,percent,change
10,change|30,Change %,percent,change
11,change|60,Change %,percent,change


In [82]:
processed_fields_pd.to_excel("trv_api_fields_processed.xlsx", index=False)